In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import missingno as msno 
import matplotlib.pyplot as plt
import plotly.express as px

In [8]:
dane = pd.read_csv('dataset.csv', sep =",")

In [10]:
print(dane.columns)



Index(['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name',
       'popularity', 'duration_ms', 'explicit', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'track_genre'],
      dtype='str')


In [12]:
import pandas as pd
import numpy as np

def validate_spotify_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Sprawdza logiczną i dziedzinową poprawność danych ze Spotify API.
    Zwraca DataFrame z podsumowaniem wykrytych anomalii.
    """
    issues = []
    total_rows = len(df)

    def log_issue(column, rule_desc, condition_mask):
        # Zliczamy tylko tam, gdzie warunek błędu jest True (ignorując NaN, chyba że sprawdzamy NaN)
        count = condition_mask.fillna(False).sum()
        if count > 0:
            issues.append({
                "Kolumna": column,
                "Wykryty problem": rule_desc,
                "Liczba błędów": count,
                "Procent danych": f"{(count / total_rows) * 100:.2f}%"
            })

    # 1. Braki danych i duplikaty w kluczowych kolumnach
    for col in ['track_id', 'artists', 'track_name', 'track_genre']:
        if col in df.columns:
            log_issue(col, "Brak wartości (NaN)", df[col].isna())
            if df[col].dtype == object:
                log_issue(col, "Pusty ciąg znaków lub same spacje", df[col].astype(str).str.strip() == "")

    if 'track_id' in df.columns:
        # Uwaga: w tym zbiorze ten sam utwór może występować pod różnymi gatunkami (track_genre)
        dup_exact = df.duplicated(subset=['track_id', 'track_genre'])
        log_issue("track_id + track_genre", "Dokładny duplikat utworu w tym samym gatunku", dup_exact)

    # 2. Metadane liczbowe i logiczne
    if 'popularity' in df.columns:
        log_issue("popularity", "Wartość poza zakresem [0, 100]", (df['popularity'] < 0) | (df['popularity'] > 100))

    if 'duration_ms' in df.columns:
        log_issue("duration_ms", "Czas <= 0 ms", df['duration_ms'] <= 0)
        log_issue("duration_ms", "Podejrzanie krótki utwór (< 10 sek.)", (df['duration_ms'] > 0) & (df['duration_ms'] < 10_000))
        log_issue("duration_ms", "Podejrzanie długi utwór (> 2 godz.)", df['duration_ms'] > 7_200_000)

    if 'explicit' in df.columns:
        valid_explicit = df['explicit'].isin([True, False, 0, 1])
        log_issue("explicit", "Wartość inna niż True/False", ~valid_explicit)

    # 3. Cechy audio w znormalizowanej skali [0.0, 1.0]
    prob_cols = [
        'danceability', 'energy', 'speechiness', 
        'acousticness', 'instrumentalness', 'liveness', 'valence'
    ]
    for col in prob_cols:
        if col in df.columns:
            log_issue(col, "Wartość poza zakresem [0.0, 1.0]", (df[col] < 0.0) | (df[col] > 1.0))

    # 4. Parametry muzyczne i fizyczne
    if 'key' in df.columns:
        # Spotify koduje tonacje jako liczby całkowite od 0 (C) do 11 (B), czasem -1 gdy brak detekcji
        valid_keys = list(range(-1, 12))
        log_issue("key", "Tonacja spoza standardu Pitch Class (-1 do 11)", ~df['key'].isin(valid_keys))

    if 'mode' in df.columns:
        # 0 = moll (minor), 1 = dur (major)
        log_issue("mode", "Skala inna niż 0 (minor) lub 1 (major)", ~df['mode'].isin([0, 1]))

    if 'loudness' in df.columns:
        # Skala dBFS: 0 dB to maksymalny sygnał cyfrowy (wartości > 0 oznaczają przesterowanie/błąd)
        log_issue("loudness", "Głośność > 0 dB (niemożliwe w skali dBFS)", df['loudness'] > 0)
        log_issue("loudness", "Głośność < -60 dB (praktycznie absolutna cisza)", df['loudness'] < -60)

    if 'tempo' in df.columns:
        log_issue("tempo", "Tempo <= 0 BPM (brak rytmu lub błąd algorytmu)", df['tempo'] <= 0)
        log_issue("tempo", "Tempo nierealistycznie wysokie (> 250 BPM)", df['tempo'] > 250)

    if 'time_signature' in df.columns:
        # Spotify standardowo szacuje metrum od 3 (3/4) do 7 (7/4); czasem pojawia się 1 lub 0 przy błędach
        log_issue("time_signature", "Nietypowe lub błędne metrum (spoza [3, 7])", ~df['time_signature'].isin([3, 4, 5, 6, 7]))

    # 5. Sprawdzenie kolumny śmieciowej
    if 'Unnamed: 0' in df.columns:
        log_issue("Unnamed: 0", "Zbędna kolumna indeksowa z pliku CSV (zalecane usunięcie)", pd.Series(True, index=df.index))

    # Zwrócenie raportu
    if not issues:
        print("✅ Wszystkie sprawdzone zmienne mają logiczne wartości!")
        return pd.DataFrame()
    
    return pd.DataFrame(issues)

# --- PRZYKŁAD UŻYCIA ---
raport = validate_spotify_dataset(dane)
print(raport.to_string(index=False))

               Kolumna                                           Wykryty problem  Liczba błędów Procent danych
               artists                                       Brak wartości (NaN)              1          0.00%
            track_name                                       Brak wartości (NaN)              1          0.00%
track_id + track_genre              Dokładny duplikat utworu w tym samym gatunku            450          0.39%
           duration_ms                                              Czas <= 0 ms              1          0.00%
           duration_ms                      Podejrzanie krótki utwór (< 10 sek.)              1          0.00%
              loudness                 Głośność > 0 dB (niemożliwe w skali dBFS)             90          0.08%
                 tempo            Tempo <= 0 BPM (brak rytmu lub błąd algorytmu)            157          0.14%
        time_signature                Nietypowe lub błędne metrum (spoza [3, 7])           1136          1.00%
 